## Clean Interactive Code with Input Prompt 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ============================================
# FUNCTION 1: Load Data from File Path
# ============================================
def load_data(file_path, file_type=None, **kwargs):
    """
    Load data from various file formats based on file path
    """
    if file_path is None or file_path.strip() == "":
        raise ValueError("❌ File path cannot be empty. Please provide a valid file path.")
    
    file_path = Path(file_path.strip())
    
    if not file_path.exists():
        raise FileNotFoundError(f"❌ File not found: {file_path}")
    
    # Infer file type from extension if not specified
    if file_type is None:
        file_type = file_path.suffix.lower().replace('.', '')
    
    print(f"\n📂 Loading file: {file_path.name}")
    print(f"📁 File type: {file_type}")
    
    try:
        # Load based on file type
        if file_type in ['csv', 'tsv', 'txt']:
            if file_type == 'csv':
                df = pd.read_csv(file_path, **kwargs)
            elif file_type == 'tsv':
                df = pd.read_csv(file_path, sep='\t', **kwargs)
            else:
                df = pd.read_csv(file_path, sep=None, engine='python', **kwargs)
                
        elif file_type in ['xlsx', 'xls']:
            df = pd.read_excel(file_path, **kwargs)
            
        elif file_type == 'json':
            df = pd.read_json(file_path, **kwargs)
            
        elif file_type == 'parquet':
            df = pd.read_parquet(file_path, **kwargs)
            
        elif file_type == 'feather':
            df = pd.read_feather(file_path, **kwargs)
            
        elif file_type == 'pickle' or file_type == 'pkl':
            df = pd.read_pickle(file_path, **kwargs)
            
        else:
            raise ValueError(f"Unsupported file type: {file_type}")
        
        print(f"✅ Successfully loaded {len(df)} rows and {len(df.columns)} columns")
        print(f"📋 Columns: {df.columns.tolist()}")
        return df
        
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        raise

# ============================================
# FUNCTION 2: Visualize Detection Limits
# ============================================
def visualize_detection_limits(df, lod_value=0.5, column=None):
    """
    Automatically detect and visualize detection limits in numeric columns
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not numeric_cols:
        print("⚠️ No numeric columns found in the dataset!")
        return
    
    if column is not None and column in numeric_cols:
        col_to_plot = column
    else:
        col_to_plot = numeric_cols[0]
        if column is not None:
            print(f"⚠️ Column '{column}' not found or not numeric. Using '{col_to_plot}' instead")
    
    print(f"\n🔍 Visualizing detection limits for column: '{col_to_plot}'")
    
    plt.figure(figsize=(14, 6))
    
    # Before handling
    plt.subplot(1, 2, 1)
    clean_data = df[col_to_plot].replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(clean_data) > 0:
        sns.histplot(clean_data, bins=30, kde=True)
        plt.title(f'Before Detection Limit Handling\nColumn: {col_to_plot}')
        plt.axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        plt.xlabel(col_to_plot)
        plt.ylabel('Frequency')
        plt.legend()
        
        stats_text = f'Mean: {clean_data.mean():.2f}\nMedian: {clean_data.median():.2f}\nStd: {clean_data.std():.2f}\nN: {len(clean_data)}'
        plt.text(0.98, 0.95, stats_text, transform=plt.gca().transAxes, 
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        plt.text(0.5, 0.5, 'No valid data to plot', ha='center', va='center')
        plt.title(f'Column: {col_to_plot} - No Data')
    
    # After handling
    plt.subplot(1, 2, 2)
    data_after = clean_data.copy()
    mask_below_lod = data_after < lod_value
    data_after[mask_below_lod] = lod_value / 2
    
    if len(data_after) > 0:
        sns.histplot(data_after, bins=30, kde=True, color='green')
        plt.title(f'After Detection Limit Handling\n(LOD replaced with LOD/2)')
        plt.axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        plt.axvline(x=lod_value/2, color='orange', linestyle='--', linewidth=2, label=f'LOD/2 = {lod_value/2}')
        plt.xlabel(col_to_plot)
        plt.ylabel('Frequency')
        plt.legend()
        
        stats_text_after = f'Mean: {data_after.mean():.2f}\nMedian: {data_after.median():.2f}\nStd: {data_after.std():.2f}\nN: {len(data_after)}'
        plt.text(0.98, 0.95, stats_text_after, transform=plt.gca().transAxes, 
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
        
        affected_count = mask_below_lod.sum()
        total_count = len(data_after)
        plt.text(0.02, 0.95, f'Values below LOD: {affected_count}/{total_count} ({affected_count/total_count*100:.1f}%)',
                transform=plt.gca().transAxes, verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    else:
        plt.text(0.5, 0.5, 'No valid data after processing', ha='center', va='center')
        plt.title(f'Column: {col_to_plot} - No Data')
    
    plt.tight_layout()
    plt.show()
    
    return numeric_cols

# ============================================
# FUNCTION 3: Complete EDA Visualization
# ============================================
def complete_eda_visualization(df, output_dir=None):
    """
    Complete EDA visualization showing all important aspects
    """
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Data Overview
    ax1 = plt.subplot(3, 3, 1)
    ax1.text(0.5, 0.8, f"Dataset Shape: {df.shape}", ha='center', fontsize=12)
    ax1.text(0.5, 0.6, f"Missing Values: {df.isnull().sum().sum()}", ha='center', fontsize=12)
    ax1.text(0.5, 0.4, f"Duplicate Rows: {df.duplicated().sum()}", ha='center', fontsize=12)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    ax1.text(0.5, 0.2, f"Numeric Columns: {len(numeric_cols)}", ha='center', fontsize=12)
    ax1.set_title('Data Overview')
    ax1.axis('off')
    
    # 2. Missing Values Heatmap
    ax2 = plt.subplot(3, 3, 2)
    if df.isnull().sum().sum() > 0:
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis', ax=ax2)
        ax2.set_title('Missing Value Pattern')
    else:
        ax2.text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
        ax2.set_title('Missing Value Pattern')
        ax2.axis('off')
    
    # 3. Numeric Columns Distribution
    ax3 = plt.subplot(3, 3, 3)
    if numeric_cols:
        first_col = numeric_cols[0]
        clean_data = df[first_col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(clean_data) > 0:
            sns.histplot(clean_data, kde=True, ax=ax3)
            ax3.set_title(f'Distribution of {first_col}')
            ax3.set_xlabel(first_col)
    
    # 4. Correlation Matrix
    ax4 = plt.subplot(3, 3, 4)
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr()
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
                   square=True, ax=ax4, cbar_kws={'shrink': 0.8})
        ax4.set_title('Correlation Matrix')
    
    # 5. Box Plots for Numeric Columns
    ax5 = plt.subplot(3, 3, 5)
    if numeric_cols:
        cols_to_plot = numeric_cols[:min(5, len(numeric_cols))]
        data_for_box = df[cols_to_plot].replace([np.inf, -np.inf], np.nan)
        if len(data_for_box.columns) > 0:
            data_for_box.boxplot(ax=ax5, rot=45)
            ax5.set_title('Boxplots of Numeric Columns')
    
    # 6. Value Counts for Categorical Columns
    ax6 = plt.subplot(3, 3, 6)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_cols:
        first_cat = cat_cols[0]
        top_categories = df[first_cat].value_counts().head(10)
        if len(top_categories) > 0:
            top_categories.plot(kind='barh', ax=ax6)
            ax6.set_title(f'Top Values in {first_cat}')
            ax6.set_xlabel('Count')
    
    # 7. Scatter Plot
    ax7 = plt.subplot(3, 3, 7)
    if len(numeric_cols) >= 2:
        col1, col2 = numeric_cols[0], numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0]
        if col1 != col2:
            scatter_data = df[[col1, col2]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(scatter_data) > 0:
                sns.scatterplot(data=scatter_data, x=col1, y=col2, ax=ax7, alpha=0.5)
                ax7.set_title(f'{col1} vs {col2}')
    
    # 8. Data Types Distribution
    ax8 = plt.subplot(3, 3, 8)
    dtypes = df.dtypes.value_counts()
    dtypes.plot(kind='pie', ax=ax8, autopct='%1.1f%%')
    ax8.set_title('Data Types Distribution')
    ax8.set_ylabel('')
    
    # 9. Memory Usage
    ax9 = plt.subplot(3, 3, 9)
    memory_usage = df.memory_usage(deep=True)
    top_memory = memory_usage.sort_values(ascending=False).head(10)
    if len(top_memory) > 1:
        top_memory[1:].plot(kind='barh', ax=ax9)
        ax9.set_title('Memory Usage by Column')
        ax9.set_xlabel('Memory (bytes)')
    
    plt.tight_layout()
    
    if output_dir:
        fig_path = output_dir / 'eda_visualization.png'
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"💾 Figure saved to: {fig_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("📊 DATASET SUMMARY")
    print("="*60)
    print(f"Total Rows: {len(df)}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Missing Values: {df.isnull().sum().sum()}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print(f"Numeric Columns: {len(numeric_cols)}")
    print(f"Categorical Columns: {len(cat_cols)}")
    
    if numeric_cols:
        print("\n📈 Numeric Column Statistics:")
        print(df[numeric_cols].describe().to_string())
    
    if output_dir:
        summary_path = output_dir / 'data_summary.txt'
        with open(summary_path, 'w') as f:
            f.write("DATA SUMMARY\n")
            f.write("="*60 + "\n")
            f.write(f"Total Rows: {len(df)}\n")
            f.write(f"Total Columns: {len(df.columns)}\n")
            f.write(f"Missing Values: {df.isnull().sum().sum()}\n")
            f.write(f"Duplicate Rows: {df.duplicated().sum()}\n")
            f.write(f"Numeric Columns: {len(numeric_cols)}\n")
            f.write(f"Categorical Columns: {len(cat_cols)}\n\n")
            if numeric_cols:
                f.write("Numeric Column Statistics:\n")
                f.write(df[numeric_cols].describe().to_string())
        print(f"💾 Summary saved to: {summary_path}")

# ============================================
# FUNCTION 4: Interactive Detection Limits (Jupyter only)
# ============================================
def interactive_detection_limits(df):
    """
    Interactive visualization with column selection (Jupyter only)
    """
    try:
        from ipywidgets import interact, widgets
    except ImportError:
        print("⚠️ ipywidgets not installed. Run: pip install ipywidgets")
        return
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not numeric_cols:
        print("No numeric columns found!")
        return
    
    def plot_column(col_name, lod_value=0.5, handling_method='LOD/2'):
        clean_data = df[col_name].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(clean_data) == 0:
            print(f"No valid data in column: {col_name}")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        axes[0].hist(clean_data, bins=30, alpha=0.7, edgecolor='black')
        axes[0].axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        axes[0].set_title(f'Before Handling\n{col_name}')
        axes[0].set_xlabel(col_name)
        axes[0].set_ylabel('Frequency')
        axes[0].legend()
        
        data_after = clean_data.copy()
        if handling_method == 'LOD/2':
            data_after[data_after < lod_value] = lod_value / 2
            label = f'LOD/2 = {lod_value/2}'
        elif handling_method == 'LOD/√2':
            data_after[data_after < lod_value] = lod_value / np.sqrt(2)
            label = f'LOD/√2 = {lod_value/np.sqrt(2):.3f}'
        else:
            data_after = data_after[data_after >= lod_value]
            label = f'Remove < LOD'
        
        axes[1].hist(data_after, bins=30, alpha=0.7, color='green', edgecolor='black')
        axes[1].axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        if handling_method in ['LOD/2', 'LOD/√2']:
            axes[1].axvline(x=lod_value/2, color='orange', linestyle='--', 
                          linewidth=2, label=label)
        axes[1].set_title(f'After Handling\n{col_name}')
        axes[1].set_xlabel(col_name)
        axes[1].set_ylabel('Frequency')
        axes[1].legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n📊 Statistics for '{col_name}':")
        print(f"Before: Mean={clean_data.mean():.3f}, Median={clean_data.median():.3f}, "
              f"Std={clean_data.std():.3f}")
        print(f"After:  Mean={data_after.mean():.3f}, Median={data_after.median():.3f}, "
              f"Std={data_after.std():.3f}")
        if isinstance(data_after, pd.Series):
            print(f"Values changed: {len(clean_data) - len(data_after)} ({100*(1-len(data_after)/len(clean_data)):.1f}%)")
    
    interact(plot_column,
             col_name=widgets.Dropdown(options=numeric_cols, description='Column:'),
             lod_value=widgets.FloatSlider(min=0.1, max=10, step=0.1, value=0.5, description='LOD Value:'),
             handling_method=widgets.Dropdown(options=['LOD/2', 'LOD/√2', 'Remove'], 
                                             description='Method:'))

# ============================================
# MAIN EXECUTION - Interactive Prompts
# ============================================

def main():
    """
    Main function that interactively prompts for file path and runs analysis
    """
    print("="*70)
    print("🚀 DATA ANALYSIS PIPELINE")
    print("="*70)
    print("\nThis tool will help you analyze your dataset with:")
    print("  • Detection limits visualization")
    print("  • Complete Exploratory Data Analysis (EDA)")
    print("  • Data quality metrics and statistics")
    print("\n" + "="*70)
    
    # Get file path from user
    print("\n📂 Please enter the path to your data file:")
    print("   (Examples: C:\\Users\\YourName\\data.csv or /home/user/data.xlsx)")
    
    while True:
        file_path = input("\n👉 File path: ").strip()
        
        if file_path == "":
            print("❌ File path cannot be empty. Please try again.")
            continue
        
        # Remove quotes if present
        file_path = file_path.strip('"').strip("'")
        
        # Check if file exists
        if not Path(file_path).exists():
            print(f"❌ File not found: {file_path}")
            print("   Please check the path and try again.")
            retry = input("   Try again? (y/n): ").strip().lower()
            if retry != 'y':
                print("❌ Analysis cancelled.")
                return
            continue
        
        break
    
    # Ask for LOD value
    print("\n📊 Please enter the Limit of Detection (LOD) value:")
    print("   (Default: 0.5)")
    lod_input = input("👉 LOD value (press Enter for default): ").strip()
    lod_value = float(lod_input) if lod_input else 0.5
    
    # Ask for specific column (optional)
    print("\n📊 Would you like to specify a column for detection limits visualization?")
    print("   (Press Enter to auto-detect numeric columns)")
    column = input("👉 Column name (optional): ").strip()
    column = column if column else None
    
    # Ask for output directory
    print("\n💾 Would you like to save the results?")
    print("   (Press Enter for default 'analysis_results' folder)")
    output_dir = input("👉 Output directory (optional): ").strip()
    output_dir = output_dir if output_dir else "analysis_results"
    
    # Ask for analysis type
    print("\n🔍 What would you like to do?")
    print("   1. Full Analysis (Load + Detection Limits + Complete EDA)")
    print("   2. Just Load Data and Show Summary")
    print("   3. Detection Limits Visualization Only")
    print("   4. Complete EDA Only")
    print("   5. Interactive Visualization (Jupyter only)")
    print("   6. Custom Analysis (load data and explore manually)")
    
    choice = input("\n👉 Enter your choice (1-6): ").strip()
    
    try:
        # Load data
        print("\n" + "="*70)
        df = load_data(file_path)
        
        if df is None or df.empty:
            print("❌ No data loaded. Exiting.")
            return
        
        # Print first few rows
        print("\n📋 First 5 rows of your data:")
        print(df.head())
        
        # Execute based on choice
        if choice == "1":
            print("\n" + "="*70)
            print("🔍 Running Full Analysis...")
            print("="*70)
            visualize_detection_limits(df, lod_value, column)
            complete_eda_visualization(df, output_dir)
            
        elif choice == "2":
            print("\n" + "="*70)
            print("📊 Data Summary")
            print("="*70)
            print(f"\nShape: {df.shape}")
            print(f"\nColumns: {df.columns.tolist()}")
            print(f"\nData Types:\n{df.dtypes}")
            print(f"\nMissing Values:\n{df.isnull().sum()}")
            print(f"\nStatistical Summary:\n{df.describe()}")
            
        elif choice == "3":
            print("\n" + "="*70)
            print("🔍 Detection Limits Visualization")
            print("="*70)
            visualize_detection_limits(df, lod_value, column)
            
        elif choice == "4":
            print("\n" + "="*70)
            print("📊 Complete EDA")
            print("="*70)
            complete_eda_visualization(df, output_dir)
            
        elif choice == "5":
            print("\n" + "="*70)
            print("🎮 Interactive Visualization")
            print("="*70)
            interactive_detection_limits(df)
            
        elif choice == "6":
            print("\n" + "="*70)
            print("🔧 Custom Analysis Mode")
            print("="*70)
            print("\n✅ Data loaded successfully!")
            print(f"   Shape: {df.shape}")
            print(f"   Columns: {df.columns.tolist()}")
            print("\n💡 You can now explore the data manually using the 'df' variable")
            print("   Example commands:")
            print("   - df.head()")
            print("   - df.info()")
            print("   - df.describe()")
            print("   - df['column_name'].hist()")
            
        else:
            print("❌ Invalid choice. Please run again and select 1-6.")
            
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("\nPlease check your file and try again.")

# ============================================
# RUN THE PROGRAM
# ============================================

if __name__ == "__main__":
    main()

🚀 DATA ANALYSIS PIPELINE

This tool will help you analyze your dataset with:
  • Detection limits visualization
  • Complete Exploratory Data Analysis (EDA)
  • Data quality metrics and statistics


📂 Please enter the path to your data file:
   (Examples: C:\Users\YourName\data.csv or /home/user/data.xlsx)



👉 File path:  all_new_data(all_new_data).csv



📊 Please enter the Limit of Detection (LOD) value:
   (Default: 0.5)


👉 LOD value (press Enter for default):  



📊 Would you like to specify a column for detection limits visualization?
   (Press Enter to auto-detect numeric columns)


...


💾 Would you like to save the results?
   (Press Enter for default 'analysis_results' folder)


👉 Output directory (optional):  



🔍 What would you like to do?
   1. Full Analysis (Load + Detection Limits + Complete EDA)
   2. Just Load Data and Show Summary
   3. Detection Limits Visualization Only
   4. Complete EDA Only
   5. Interactive Visualization (Jupyter only)
   6. Custom Analysis (load data and explore manually)



👉 Enter your choice (1-6):  1




📂 Loading file: all_new_data(all_new_data).csv
📁 File type: csv
❌ Error loading file: 'utf-8' codec can't decode byte 0xe1 in position 526: invalid continuation byte

❌ Error: 'utf-8' codec can't decode byte 0xe1 in position 526: invalid continuation byte

Please check your file and try again.
